# Variables Antitéticas

---
Nombre: **Alan Jesús Hernández Soto**  
Carrera: **Ingeniería Financiera**  
Materia: **Simulación de Procesos Financieros**  
Tipo de trabajo: **Clase**

---

## Introducción

Exploramos el uso de variables antitéticas para estimar el valor de una opción call europea. La idea es combinar cada choque normal $z$ con su opuesto $-z$ y promediar sus pagos para reducir la variabilidad de la estimación.

La función `bs` calcula el precio teórico con Black-Scholes y servirá como referencia; con los parámetros del ejercicio, la call vale aproximadamente **10.4506**. Para ejecutar esta celda es necesario haber importado las librerías y definido los parámetros de las celdas siguientes.

In [48]:

def bs(S0, K, r, sigma, T, b_a, call_opt=True):
    d1 = (np.log(S0 / K) + (b_a + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    put = K * np.exp(-r * T) * norm.cdf(-d2) - S0 * np.exp((b_a - r)*T) * norm.cdf(-d1)
    call = S0 * np.exp((b_a - r) * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    if call_opt:
        return call
    else:
        return put

bs_call = black_scholes_call = bs(S0=s0, K=K, r=r, sigma=sigma, T=T, b_a=r, call_opt=True)
bs_call

np.float64(10.450583572185565)

## Librerías y simetría de la normal

Importamos las herramientas de cálculo, tablas y distribuciones. Generamos 10,000 observaciones normales estándar y comparamos las medias de los valores positivos y negativos para observar su simetría; esta propiedad permite formar pares $z$ y $-z$.

In [49]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import norm

zval = np.random.normal(0, 1, 10000)

zval_pos = zval[zval > 0].mean()
zval_neg = zval[zval < 0].mean()

zval_pos, zval_neg

(np.float64(0.7983030624787114), np.float64(-0.8005065440510909))

## Parámetros y pago de la call

Usamos un precio inicial y de ejercicio de 100, tasa anual de 5%, volatilidad de 20% y vencimiento a un año. Simulamos el precio final mediante un movimiento browniano geométrico y calculamos el pago $\max(S_T-K,0)$. Con el choque $z=-1$, la opción termina sin valor y su pago es cero.

In [50]:
s0 = 100
r = 0.05
sigma = 0.2
T = 1
K = 100

call_z1 = np.maximum(s0 * np.exp((r - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * -1) - K, 0)
call_z1

np.float64(0.0)

Usamos diferentes valores de Z para calcular y promediar los valores de call.

In [51]:
z = np.array([0.07629, -0.2985, -0.7983, 0.993, -1.3465, 1.2462, -1.1405, 0.7615, 0.2816, 0.81])



call_z = np.maximum(0, s0 * np.exp((r - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * z) - K)
call_z_antithetic = np.maximum(0, s0 * np.exp((r - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * -z) - K)

values = pd.DataFrame({'z': z, 'call_z': call_z, 'call_z_antithetic': call_z_antithetic})
values['call'] = values[['call_z', 'call_z_antithetic']].mean(axis=1)
display(values)

call_mean = values['call'].mean()
print(f'Promedio de la call con z positivas {values["call_z"].mean():.4f}')
print(f'Promedio de la call con z negativas {values["call_z_antithetic"].mean():.4f}')
print(f'Valor promedio de la opción call con estas variables antitéticas: {call_mean:.4f}')



,z,call_z,call_z_antithetic,call
0,0.07629,4.629777,1.485120,3.057448
1,-0.29850,0.000000,9.384608,4.692304
2,-0.79830,0.000000,20.883852,10.441926
3,0.99300,25.683920,0.000000,12.841960
4,-1.34650,0.000000,34.891424,17.445712
5,1.24620,32.212462,0.000000,16.106231
6,-1.14050,0.000000,29.446826,14.723413
7,0.76150,19.997413,0.000000,9.998707
8,0.28160,9.015512,0.000000,4.507756
9,0.81000,21.167052,0.000000,10.583526


Promedio de la call con z positivas 11.2706
Promedio de la call con z negativas 9.6092
Valor promedio de la opción call con estas variables antitéticas: 10.4399
